In [3]:
import threading
import time

def download_data(task_id):
    print(f"[Thread {task_id}] Starting download...")
    time.sleep(2)
    print(f"[Thread {task_id}] Download complete!")

def main():
    threads = []
    start = time.time()

    for i in range(5):
        t = threading.Thread(target=download_data, args=(i,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print("All threads finished in", round(time.time() - start, 2), "seconds")

if __name__ == "__main__":
    main()


[Thread 0] Starting download...
[Thread 1] Starting download...
[Thread 2] Starting download...
[Thread 3] Starting download...
[Thread 4] Starting download...
[Thread 1] Download complete![Thread 0] Download complete!
[Thread 2] Download complete!
[Thread 4] Download complete!

[Thread 3] Download complete!
All threads finished in 2.04 seconds


In [4]:
from time import sleep, perf_counter
from threading import Thread


def task(id):
    print(f'Starting the task {id}...')
    sleep(1)
    print(f'The task {id} completed')


start_time = perf_counter()

# create and start 10 threads
threads = []
for n in range(1, 11):
    t = Thread(target=task, args=(n,))
    threads.append(t)
    t.start()

# wait for the threads to complete
for t in threads:
    t.join()

end_time = perf_counter()

print(f'It took {end_time- start_time: 0.2f} second(s) to complete.')

Starting the task 1...
Starting the task 2...
Starting the task 3...
Starting the task 4...
Starting the task 5...
Starting the task 6...
Starting the task 7...
Starting the task 8...
Starting the task 9...
Starting the task 10...
The task 1 completedThe task 2 completed

The task 3 completed
The task 4 completed
The task 5 completed
The task 6 completed
The task 7 completed
The task 8 completed
The task 10 completed
The task 9 completed
It took  1.01 second(s) to complete.


In [9]:
from threading import Thread
import urllib.request
import urllib.error
import http.client


class HttpRequestThread(Thread):
    def __init__(self, url: str) -> None:
        super().__init__()
        self.url = url

    def run(self) -> None:
        print(f'Checking {self.url} ...')
        try:
            response = urllib.request.urlopen(self.url)
            print(response.code)

        except urllib.error.HTTPError as e:
            # HTTP status errors (400, 500, etc.)
            self.http_status_code = e.code
            self.reason = e.reason

        except urllib.error.URLError as e:
            # DNS / connection issues
            self.reason = e.reason

        except http.client.RemoteDisconnected as e:
            # Server closed connection abruptly
            self.reason = "RemoteDisconnected"

        except Exception as e:
            # Catch-all safety net
            self.reason = str(e)

def main() -> None:
    urls = [
        'https://httpstat.us/200',
        'https://httpstat.us/400'
    ]

    threads = [HttpRequestThread(url) for url in urls]

    [t.start() for t in threads]

    [t.join() for t in threads]


if __name__ == '__main__':
    main()

Checking https://httpstat.us/200 ...
Checking https://httpstat.us/400 ...


In [8]:
from threading import Thread
import urllib.request
import urllib.error
import http.client


class HttpRequestThread(Thread):
    def __init__(self, url: str) -> None:
        super().__init__()
        self.url = url
        self.http_status_code = None
        self.reason = None

    def run(self) -> None:
        try:
            response = urllib.request.urlopen(self.url, timeout=5)
            self.http_status_code = response.getcode()

        except urllib.error.HTTPError as e:
            # HTTP status errors (400, 500, etc.)
            self.http_status_code = e.code
            self.reason = e.reason

        except urllib.error.URLError as e:
            # DNS / connection issues
            self.reason = e.reason

        except http.client.RemoteDisconnected as e:
            # Server closed connection abruptly
            self.reason = "RemoteDisconnected"

        except Exception as e:
            # Catch-all safety net
            self.reason = str(e)


def main() -> None:
    urls = [
        "https://httpstat.us/200",
        "https://httpstat.us/400",
    ]

    threads = [HttpRequestThread(url) for url in urls]

    for t in threads:
        t.start()

    for t in threads:
        t.join()

    for t in threads:
        if t.http_status_code is not None:
            print(f"{t.url}: {t.http_status_code}")
        else:
            print(f"{t.url}: ERROR - {t.reason}")


if __name__ == "__main__":
    main()


https://httpstat.us/200: ERROR - RemoteDisconnected
https://httpstat.us/400: ERROR - RemoteDisconnected


# https://www.pythontutorial.net/

In [ ]:
class File:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        print(f'Opening the file {self.filename}.')
        self.__file = open(self.filename, self.mode)
        return self.__file

    def __exit__(self, exc_type, exc_value, exc_traceback):
        print(f'Closing the file {self.filename}.')
        if not self.__file.closed:
            self.__file.close()

        return False


with File('data.txt', 'r') as f:
    print(int(next(f)))

# ThreadPoolExecutor 
"""
 The ThreadPoolExecutor class extends the Executor class and returns a Future object.

Executor 
The Executor class has three methods to control the thread pool:

submit() – dispatch a function to be executed and return a Future object. The submit() method takes a function and executes it asynchronously.
map() – execute a function asynchronously for each element in an iterable.
shutdown() – shut down the executor.
When you create a new instance of the ThreadPoolExecutor class, Python starts the Executor.

Once completing working with the executor, you must explicitly call the shutdown() method to release the resource held by the executor. To avoid calling the shutdown() method explicitly, you can use the context manager.

Future object 
A Future is an object that represents the eventual result of an asynchronous operation. The Future class has two useful methods:

result() – return the result of an asynchronous operation.
exception() – return the exception of an asynchronous operation in case an exception occurs."""


In [ ]:
from time import sleep, perf_counter
from concurrent.futures import ThreadPoolExecutor

def task(id):
    print(f'Starting the task {id}...')
    sleep(1)
    return f'Done with task {id}'

start = perf_counter()

with ThreadPoolExecutor() as executor:
    f1 = executor.submit(task, 1)
    f2 = executor.submit(task, 2)

    print(f1.result())
    print(f2.result())    

finish = perf_counter()

print(f"It took {finish-start} second(s) to finish.")ss

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen
import time
import os

def download_image(url):
    image_data = None
    with urlopen(url) as f:
        image_data = f.read()

    if not image_data:
        raise Exception(f"Error: could not download the image from {url}")

    filename = os.path.basename(url)
    with open(filename, 'wb') as image_file:
        image_file.write(image_data)
        print(f'{filename} was downloaded...')

start = time.perf_counter()

urls = ['https://upload.wikimedia.org/wikipedia/commons/9/9d/Python_bivittatus_1701.jpg',
        'https://upload.wikimedia.org/wikipedia/commons/4/48/Python_Regius.jpg',
        'https://upload.wikimedia.org/wikipedia/commons/d/d3/Baby_carpet_python_caudal_luring.jpg',
        'https://upload.wikimedia.org/wikipedia/commons/f/f0/Rock_python_pratik.JPG',
        'https://upload.wikimedia.org/wikipedia/commons/0/07/Dulip_Wilpattu_Python1.jpg']

with ThreadPoolExecutor() as executor:
      executor.map(download_image, urls)

finish = time.perf_counter()    

print(f'It took {finish-start} second(s) to finish.')

# Race condition 

In [19]:
from threading import Thread
from time import sleep


counter = 0

def increase(by):
    global counter

    local_counter = counter
    local_counter += by

    sleep(0.1)

    counter = local_counter
    print(f'counter={counter}')


# create threads
t1 = Thread(target=increase, args=(10,))
t2 = Thread(target=increase, args=(20,))

# start the threads
t1.start()
t2.start()


# wait for the threads to complete
t1.join()
t2.join()


print(f'The final counter is {counter}')

counter=20
counter=10
The final counter is 10


In [20]:
from threading import Thread, Lock
from time import sleep


counter = 0


def increase(by, lock):
    global counter

    lock.acquire()

    local_counter = counter
    local_counter += by

    sleep(0.1)

    counter = local_counter
    print(f'counter={counter}')

    lock.release()


lock = Lock()

# create threads
t1 = Thread(target=increase, args=(10, lock))
t2 = Thread(target=increase, args=(20, lock))

# start the threads
t1.start()
t2.start()


# wait for the threads to complete
t1.join()
t2.join()


print(f'The final counter is {counter}')

counter=10
counter=30
The final counter is 30


In [ ]:
from threading import Thread, Lock
from time import sleep


counter = 0

def increase(by, lock):
    global counter

    with lock:
        local_counter = counter
        local_counter += by

        sleep(0.1)

        counter = local_counter
        print(f'counter={counter}')


lock = Lock()

# create threads
t1 = Thread(target=increase, args=(10, lock))
t2 = Thread(target=increase, args=(20, lock))

# start the threads
t1.start()
t2.start()


# wait for the threads to complete
t1.join()
t2.join()


print(f'The final counter is {counter}')


In [ ]:
from threading import Thread, Lock
from time import sleep


class Counter:
    def __init__(self):
        self.value = 0
        self.lock = Lock()

    def increase(self, by):
        with self.lock:
            current_value = self.value
            current_value += by

            sleep(0.1)

            self.value = current_value
            print(f'counter={self.value}')

def main():
    counter = Counter()
    # create threads
    t1 = Thread(target=counter.increase, args=(10, ))
    t2 = Thread(target=counter.increase, args=(20, ))

    # start the threads
    t1.start()
    t2.start()


    # wait for the threads to complete
    t1.join()
    t2.join()


    print(f'The final counter is {counter.value}')

if __name__ == '__main__':
    main()


# Semaphore dec counter if > 0 else if < 0 wait for semaphore 

In [21]:
import threading
import urllib.request

MAX_CONCURRENT_DOWNLOADS = 3
semaphore = threading.Semaphore(MAX_CONCURRENT_DOWNLOADS)

def download(url):
    with semaphore:
        print(f"Downloading {url}...")
        
        response = urllib.request.urlopen(url)
        data = response.read()
        
        print(f"Finished downloading {url}")

        return data

        

def main():
    # URLs to download
    urls = [
        'https://www.ietf.org/rfc/rfc791.txt',
        'https://www.ietf.org/rfc/rfc792.txt',
        'https://www.ietf.org/rfc/rfc793.txt',
        'https://www.ietf.org/rfc/rfc794.txt',
        'https://www.ietf.org/rfc/rfc795.txt',
    ]

    # Create threads for each download
    threads = []
    for url in urls:
        thread = threading.Thread(target=download, args=(url,))
        threads.append(thread)
        thread.start()

    # Wait for all threads to complete
    for thread in threads:
        thread.join()


if __name__ == '__main__':
    main()

Finished downloading https://www.ietf.org/rfc/rfc793.txt
Finished downloading https://www.ietf.org/rfc/rfc792.txt
Finished downloading https://www.ietf.org/rfc/rfc791.txt
Finished downloading https://www.ietf.org/rfc/rfc794.txt
Finished downloading https://www.ietf.org/rfc/rfc795.txt


# thread safe queue 

In [ ]:
import time
from queue import Empty, Queue
from threading import Thread


def producer(queue):
    for i in range(1, 6):
        print(f'Inserting item {i} into the queue')
        time.sleep(1)
        queue.put(i)


def consumer(queue):
    while True:
        try:
            item = queue.get()
        except Empty:
            continue
        else:
            print(f'Processing item {item}')
            time.sleep(2)
            queue.task_done()


def main():
    queue = Queue()

    # create a producer thread and start it
    producer_thread = Thread(
        target=producer,
        args=(queue,)
    )
    producer_thread.start()

    # create a consumer thread and start it
    consumer_thread = Thread(
        target=consumer,
        args=(queue,),
        daemon=True
    )
    consumer_thread.start()

    # wait for all tasks to be added to the queue
    producer_thread.join()

    # wait for all tasks on the queue to be completed
    queue.join()


if __name__ == '__main__':
    main()